In [1]:
%cd drive/MyDrive/Thesis/Models/PLM_embedding

/content/drive/MyDrive/Thesis/Models/PLM_embedding


In [2]:
import numpy as np
import torch
import torch.nn as nn
import os 

from model import CNN_BRNN

In [3]:
%cp -R '../../Data/test_embeddings' '/home/embeddings'

KeyboardInterrupt: ignored

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN_BRNN()
model = model.to(device)
model.load_state_dict(torch.load('./saved_model', map_location = torch.device(device)))
model.float()
model.eval()

CNN_BRNN(
  (conv1): Sequential(
    (0): Conv1d(1280, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
  )
  (conv2): Sequential(
    (0): Conv1d(1280, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
  )
  (conv3): Sequential(
    (0): Conv1d(1280, 64, kernel_size=(11,), stride=(1,), padding=(5,))
    (1): ReLU()
  )
  (brnn): GRU(192, 256, num_layers=3, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Sequential(
    (0): Linear(in_features=704, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=3, bias=True)
  )
)

In [ ]:
device

'cuda'

In [4]:
def get_prediction(embedding_input):
  seqlen = embedding_input.shape[0] - 2    #BOS, EOS tokens
  embedding_input = embedding_input.to(device)
  embedding_input = embedding_input.transpose(0,1)
  embedding_input.unsqueeze_(0)            #no batches
  embedding_input = torch.nn.functional.pad(embedding_input, (700 - (seqlen+2), 0), 'constant',  0)
  output = model(embedding_input)
  _, prediction = torch.max(output, 1)
  prediction = prediction[:, (700 - 1 - seqlen) : -1]
  prediction = prediction.numpy(force=True).astype('int8')
  assert prediction.shape[1] == seqlen , print(prediction.shape)
  return(prediction)

def get_long_prediction(embedding_input):
  length = embedding_input.shape[0] 
  embedding_input = embedding_input.to(device)
  embedding_input = embedding_input.transpose(0,1)
  embedding_input.unsqueeze_(0) # 1 , L, 1280
  predictions = []
  slices = length // 700 

  output = model(embedding_input[:,:,0:700])
  _, prediction = torch.max(output, 1)
  prediction = prediction[:, 1:]
  prediction = prediction.numpy(force=True)
  predictions.append(prediction)

  for i in range(1, slices):
    output= model(embedding_input[:,:, 700*i : 700*(i+1)])
    _, prediction = torch.max(output, 1)
    prediction = prediction.numpy(force=True)
    predictions.append(prediction)
  
  output = model(embedding_input[:,:, -700:])
  _, prediction = torch.max(output, 1)
  prediction = prediction[ : , -length%700:-1]
  prediction = prediction.numpy(force=True)
  predictions.append(prediction)

  predictions = np.concatenate(predictions, axis=None, dtype='int8')
  assert len(predictions) == length-2, print(len(predictions), length)
  return(predictions)

In [5]:
PLM_predictions = {}

# for file in os.listdir("/home/embeddings"):
#   embedding = torch.load("/home/embeddings/"+file)
for file in os.listdir("../../Data/n2"):
  embedding = torch.load("../../Data/n2/"+file)  
  if embedding.shape[0] > 700 :
    prediction = get_long_prediction(embedding)
  else:
    prediction = get_prediction(embedding)
  
  chain = file.split(".")[0]
  np.save("../../Data/"+file+".npz", prediction)
  # PLM_predictions[chain] = prediction

# np.savez("../../Data/PLM_predictions.npz", **PLM_predictions)
# np.savez("../../Data/PLM62.npz", **PLM_predictions)